---
title: Structural Variants Detected
subtitle: Putative structural variants as detected by PLACEHOLDER
date: "9999-12-31"
edit_url: null
---

In [ ]:
from harpy.report.by_chromosome import sv_by_chromosome
from harpy.report.components import StatsBox, print_html, standard_itable
from harpy.report.theme import sv_colors
from harpy.report.utilities import process_variants
import itables
import pandas as pd
from pathlib import Path
import sys
itables.init_notebook_mode(connected=True)


In [ ]:
faidx = "workflow/reference/genome.fasta.fai"
indir = "."
contigs = "default"


In [ ]:
infiles = [i.as_posix() for i in Path(indir).glob("*.bedpe")]
_sv_types = {"INV": "Inversion", "DEL": "Deletion", "DUP": "Duplication", "BND": "Breakend"}

if not list(infiles):
    print_html("No input files were found. The inputs are expected to be e.g. 'inversions.bedpe'")
    sys.exit(0)

tables = []
for i in infiles:
    _x = pd.read_table(i)
    if len(_x):
        tables.append(_x)

if not tables:
    print_html("No structural variants were detected.")
    sys.exit(0)

sv = pd.concat(tables).reset_index(drop = True)
del tables
del _x

# standardize the column names btwn Leviathan and Naibr
if "Break1" in sv.columns:
    sv = sv.drop(columns = ["Chr2"])
    sv.columns = ["Sample", "Contig", "Start", "End", 'Split Mol', 'Discordant', 'Orientation', 'Haplotype', 'Score', 'Filter', 'Type']
    sv = sv[['Sample', 'Contig', 'Start', 'End', 'Type', 'Split Mol', 'Discordant', 'Orientation', 'Haplotype', 'Score', 'Filter']]
    sv['Type'] = sv['Type'].str.title()
else:
    sv.columns = [i.replace("position_", "").replace("_", " ").title() for i in sv.columns]
    sv['Type'] = [_sv_types[i] for i in sv['Type']]


In [26]:
unique_variants = process_variants(sv, bin_size=100)
unique_variants['Samples'] = unique_variants['Samples'].str.join(', ')


This report details the structural variants identified by [LEVIATHAN](https://github.com/morispi/LEVIATHAN)
([](https://doi.org/10.1101/2021.03.25.437002)) or [NAIBR](https://github.com/pontushojer/NAIBR) ([](https://doi.org/10.1093/bioinformatics/btx712))[^bnd].

[^bnd]:
    `NAIBR` does not report breakends.

In [ ]:
_sv_counts = unique_variants['Type'].value_counts().to_dict()
boxes = StatsBox()
for k,v in _sv_types.items():
    boxes.add(_sv_counts.get(v, 0), v + "s", textcol = sv_colors(k))

boxes.render()


## Unique Variants

This table and corresponding visualization detail the overview of variants detected when consolidating variant calls of the same type within a 100bp window[^2].  The `N Samples` column is the number of samples that putatively have this variant, whereas `Samples` is a list of those samples.

[^2]:
    For example, if two deletions on the same contig had their breakpoints 20bp apart, the deletions were merged and considered duplicates of each other. The merging is only for reporting/summarizing purposes, the source data was not modified.

In [29]:
standard_itable(unique_variants, "sv.unique", fixedcols = 3)


Loading ITables v2.6.2 from the internet... (need help?)


In [ ]:
contig_lens = pd.read_table(faidx, header = None).drop(columns = [2,3,4]).rename(columns = {0 : 'Contig', 1 : 'length'})

if contigs == "default":
    _keep = contig_lens.nlargest(30, 'length')['Contig'].tolist()
else:
    _keep = contigs.split(',')

unique_variants = unique_variants[unique_variants['Contig'].isin(_keep)].reset_index(drop = True)
contig_lens = contig_lens[contig_lens['Contig'].isin(_keep)].reset_index(drop = True)

unique_variants = unique_variants.merge(contig_lens, on = 'Contig')


In [31]:
sv_by_chromosome(unique_variants, "Structural Variants Detected")


alt.Chart(...)

## All Variants
This table details all the variants detected and evidence for those detections as reported by the calling software[^3].

[^3]:
    `LEVIATHAN` and `NAIBR` report different information with their variant calls.

In [32]:
standard_itable(sv.drop(columns = ['start_bin', "end_bin"]), "sv.all", fixedcols = 1)


Loading ITables v2.6.2 from the internet... (need help?)


## Supporting Info

:::{dropdown} Variant Types
**Inversion (INV)**: A segment that broke off and reattached within the same chromosome, but in reverse orientation

**Duplication (DUP)**: A type of mutation that involves the production of one or more copies of a gene or region of a chromosome

**Deletion (DEL)**: A type of mutation that involves the loss of one or more nucleotides from a segment of DNA

**Breakend (BND)**: The endpoint of a structural variant, which can be useful to explain complex variants
:::

:::{dropdown} LEVIATHAN
`LEVIATHAN` uses split (chimeric) reads to determine variant breakpoints. Split reads occur when a part of a sequence aligns well to one region and another part aligns well to a different region, thereby 'splitting' the sequence into multiple well-aligning sections. `LEVIATHAN` reports the number of barcodes and read pair supporting a variant call. The `LEVIATHAN` algorithm works in two phases; the first phase identifies putative breakpoints using 1kb windows, and the second phase heuristically filters calls and identify the exact breakpoints in the remaining windows. If you are expecting specific structural variants for some samples that were not reported, it may be worth examining the `.candidates` file(s) to see if they were initially detected and ultimately filtered out by the software.
:::

:::{dropdown} NAIBR
Similar to `LEVIATHAN`, the `NAIBR` software uses split reads to determine variant breakpoints. Additionally, `NAIBR` requires the input alignments to be phased, which facilitates the variant breakpoint identification. `NAIBR` reports more metrics than `LEVIATHAN`, which includes the number of split molecules supporting the call, discordant read pairs, alignment orientation (for determining variant type), haplotype (genotype), a quality/likelihood score, and a heuristic filter pass/fail. 
:::